In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
from snowflake.snowpark.context import get_active_session #importing connection to python
session = get_active_session() #opening the connection to python
df = session.table("MOODMATCH.PUBLIC.CLEAN_DATA").to_pandas() #opening the table and converting it to pandas

In [ ]:
if "MOOD" not in df.columns:
    df["MOOD"] = None

In [ ]:
#Scary
df.loc[df["MOOD"] == "Scary", "MOOD"] = None
sample_scary = df[
    (df["GENRES"].str.contains("Thriller|Crime|Suspense|Horror", na=False)) &
    (~df["GENRES"].str.contains("Comedy|Family", na=False)) &
    (df["MOOD"].isna())
].sample(n=150).index
df.loc[sample_scary, "MOOD"] = "Scary"

In [ ]:
#Feel-Good
df.loc[df["MOOD"] == "Feel-Good", "MOOD"] = None
feelgood_movies = df[
    ((df["GENRES"].str.contains("Comedy", na=False)) &
     (df["GENRES"].str.contains("Romance", na=False))) &
    (~df["GENRES"].str.contains("Crime|Thriller|Suspense|Horror|Science Fiction", na=False)) &
    (df["MOOD"].isna())
].sample(n=150).index
df.loc[feelgood_movies, "MOOD"] = "Feel-Good"

In [ ]:
#Adventurous
df.loc[df["MOOD"] == "Adventurous", "MOOD"] = None
adventurous_movies = df[
    (df["GENRES"].str.contains("Adventure|Science Fiction|Action", na=False)) &
    (df["MOOD"].isna())
].sample(n=150).index
df.loc[adventurous_movies, "MOOD"] = "Adventurous"


In [ ]:
#Melancholic
df.loc[df["MOOD"] == "Melancholic", "MOOD"] = None
sad_keywords = [
    "tragedy", "struggle", "heartbreak", "bittersweet", "grief",
    "loss", "sorrow", "mourning", "despair", "loneliness",
    "death", "dying", "terminal", "illness", "funeral",
    "melancholy", "somber", "poignant", "devastating",
    "emotional", "haunting", "tragic", "hopeless",
    "unrequited", "farewell", "suffering", "abandoned"
]

pattern = "|".join(sad_keywords)

sad_movies = df[
    (df["GENRES"].str.contains("Drama", na=False)) &
    (df["OVERVIEW"].str.contains(pattern, case=False, na=False)) &
    (~df["GENRES"].str.contains("Thriller", na=False)) &
    (~df["GENRES"].str.contains("Comedy", na=False)) &
    (df["MOOD"].isna())
].sample(n=150).index
df.loc[sad_movies, "MOOD"] = "Melancholic"

In [ ]:
#Nostalgic
df.loc[df["MOOD"] == "Nostalgic", "MOOD"] = None
nostalgic_keywords = [
    "journey", "quest", "magical", "enchanted", "kingdom", "rescue",
    "friendship", "friends", "sidekick", "companion",
    "fairy tale", "princess", "prince", "wizard", "witch", "spell",
    "curse", "dragon", "castle", "magical world",
    "young", "boy", "girl", "child", "grows up", "learns",
    "brave", "courage", "believe", "dream", "imagination", "wishes",
    "beloved", "animated", "musical", "evil", "villain", "hero",
    "unlikely hero", "destiny", "prophecy",
    "wonder", "whimsical", "toy", "toys", "talking", "animal",
    "forest", "ocean", "underwater"
]

nostalgic_pattern = "|".join(nostalgic_keywords)

nostalgic_movies = df[
    (df["GENRES"].str.contains("Animation|Family|Fantasy", na=False)) &
    (df["OVERVIEW"].str.contains(nostalgic_pattern, case=False, na=False)) &
    (~df["GENRES"].str.contains("Horror|Crime", na=False)) &
    (df["MOOD"].isna())
].sample(n=150).index
df.loc[nostalgic_movies, "MOOD"] = "Nostalgic"

In [ ]:
session.sql("USE DATABASE MOODMATCH").collect()
session.sql("USE SCHEMA PUBLIC").collect()

In [ ]:
labeled = session.table("MOODMATCH.PUBLIC.LABELED_DATA").to_pandas()
labeled["TEXT"] = labeled["GENRES"] + " " + labeled["OVERVIEW"]

In [ ]:
tfidf = TfidfVectorizer(max_features = 5000, stop_words = 'english')
matrix = tfidf.fit_transform(labeled["TEXT"])
print(matrix.shape)

In [ ]:
Y = labeled["MOOD"] #mood column
X = matrix

In [ ]:
model = LogisticRegression(solver='lbfgs', max_iter = 1000)
le = LabelEncoder()
y = le.fit_transform(Y) #creating the class nums for each mood
#train/test spil - kfold
kf = KFold(n_splits=5, shuffle=True, random_state=30)
#Now running the model through the five folds and getting accuracy score
score = cross_val_score(model, X, y, cv=kf)
print(score.mean())





In [ ]:
#training on entire data set
model.fit(X,y)

In [ ]:
#Pulling clean data which is df and has already been pulled
#Create text column in df
df["TEXT"] = df["GENRES"] + " " + df["OVERVIEW"]
final_matrix = tfidf.transform(df["TEXT"])
p = model.predict(final_matrix)
mood_name = le.inverse_transform(p)
df["MOOD"] = df["MOOD"].fillna(pd.Series(mood_name, index=df.index))

print(df["MOOD"].isna().sum())   # should be 0
print(df["MOOD"].value_counts()) # count per mood


In [ ]:
import numpy as np

def recommend(mood, n=10):
    filter = df[df["MOOD"] == mood]
    filter = filter[filter["OVERVIEW"].str.strip() != "No overview found."]  # remove empty overviews
    filtered_matrix = final_matrix[filter.index]
    mean_vector = np.asarray(filtered_matrix.mean(axis=0))
    sim_scores = cosine_similarity(mean_vector, np.asarray(filtered_matrix.todense()))
    filter = filter.copy()
    filter["SCORES"] = sim_scores[0]
    return filter.sort_values(by="SCORES", ascending=False)[["TITLE", "GENRES", "OVERVIEW"]].head(n)
recommend("Nostalgic")